To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth your local device, follow [our guide](https://docs.unsloth.ai/get-started/install-and-update). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News


Unsloth's [Docker image](https://hub.docker.com/r/unsloth/unsloth) is here! Start training with no setup & environment issues. [Read our Guide](https://docs.unsloth.ai/new/how-to-train-llms-with-unsloth-and-docker).

[gpt-oss RL](https://docs.unsloth.ai/new/gpt-oss-reinforcement-learning) is now supported with the fastest inference & lowest VRAM. Try our [new notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/gpt-oss-(20B)-GRPO.ipynb) which creates kernels!

Introducing [Vision](https://docs.unsloth.ai/new/vision-reinforcement-learning-vlm-rl) and [Standby](https://docs.unsloth.ai/basics/memory-efficient-rl) for RL! Train Qwen, Gemma etc. VLMs with GSPO - even faster with less VRAM.

Unsloth now supports Text-to-Speech (TTS) models. Read our [guide here](https://docs.unsloth.ai/basics/text-to-speech-tts-fine-tuning).

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [1]:
%%capture
import os

!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
!hf auth login --token $HF_TOKEN

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: write).
The token `sd` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `sd`


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

In [4]:
from unsloth import FastModel
from transformers import AutoModelForSequenceClassification
import torch

# Disable fast generation for bert!
#%env UNSLOTH_DISABLE_FAST_GENERATION = 1

import random
import numpy as np
import os

seed_value = 1998

# Python's random module
random.seed(seed_value)

# NumPy
np.random.seed(seed_value)

# PyTorch
torch.manual_seed(seed_value)
torch.cuda.manual_seed(seed_value)
torch.cuda.manual_seed_all(seed_value)  # For multi-GPU

# PyTorch backends (for deterministic behavior)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Set Python hash seed (for string hashing)
os.environ['PYTHONHASHSEED'] = str(seed_value)

# Transformers library (if using Trainer)
from transformers import set_seed
set_seed(seed_value)

max_seq_length = 512 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

id2label = {0: "OR", 1: "CG"} #, 2: "love", 3: "anger",4: "fear",5: "surprise"}

label2id = {"OR": 0, "CG": 1} #, "love": 2, "anger": 3, "fear": 4, "surprise": 5}

model, tokenizer = FastModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-0.5B-Instruct", #"google/gemma-3-270m-it", #"LiquidAI/LFM2-350M", #"Pulk17/Fake-News-Detection", #"distilbert/distilbert-base-uncased-finetuned-sst-2-english",#"answerdotai/ModernBERT-large",
    auto_model = AutoModelForSequenceClassification,
    max_seq_length = max_seq_length,
    dtype = dtype,
    num_labels  = 2,
    full_finetuning = False,
    id2label=id2label,
    label2id=label2id,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2025-12-11 15:16:23.048658: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765466183.454743      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765466183.642962      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🦥 Unsloth Zoo will now patch everything to make training faster!


[unsloth_zoo.log|WARNING]Unsloth: Failed to import trl openenv: No module named 'trl.experimental'


==((====))==  Unsloth 2025.12.4: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

We now add LoRA adapters so we only need to update a small amount of parameters!


In [5]:
#model.classifier = torch.nn.Linear(768, 2).to(model.device, dtype=torch.float32)

In [6]:
model = FastModel.get_peft_model(
    model,
    r = 64, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    #target_modules = #["query", "key", "value", "dense"], #["Wqkv", "Wo", "Wi"],
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 128,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = seed_value,
    use_rslora = True,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    task_type="SEQ_CLS",
)

Unsloth: Making `model.base_model.model.model` require gradients
Unsloth: Upcasting `base_model.model.score` from float16 to float32 since it's in `modules_to_save`. Also allowing gradients.


<a name="Data"></a>
### Data Prep  
We now use the [Emotion dataset](https://huggingface.co/datasets/dair-ai/emotion) from `dair-ai`, which contains text labeled by emotion. In this example, we load the **unsplit** version and use only the first 30,000 samples.  

We then split the dataset into training (80%) and validation (20%), and apply tokenization to prepare the text for training.


In [7]:
from datasets import load_dataset

# Load the IMDB dataset
dataset = load_dataset('csv', data_files='/kaggle/input/fake reviews dataset.csv')
dataset

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['category', 'rating', 'label', 'text_'],
        num_rows: 40432
    })
})

In [8]:
"""max_len = max(len(tokenizer(text)['input_ids']) for text in dataset['train']['text_'])
print(max_len)"""

"max_len = max(len(tokenizer(text)['input_ids']) for text in dataset['train']['text_'])\nprint(max_len)"

In [9]:
"""import numpy as np

# Tokenize all texts and get their lengths
token_lengths = [len(tokenizer(text)['input_ids']) for text in dataset['train']['text_']]

# Calculate percentiles from 90 to 99
percentiles = np.percentile(token_lengths, range(90, 100))

# Display results
for i, p in enumerate(percentiles, start=90):
    print(f"{i}th percentile: {int(p)} tokens")

# Or get specific percentiles
p90 = np.percentile(token_lengths, 90)
p95 = np.percentile(token_lengths, 95)
p99 = np.percentile(token_lengths, 99)

print(f"\n90th percentile: {int(p90)}")
print(f"95th percentile: {int(p95)}")
print(f"99th percentile: {int(p99)}")"""

'import numpy as np\n\n# Tokenize all texts and get their lengths\ntoken_lengths = [len(tokenizer(text)[\'input_ids\']) for text in dataset[\'train\'][\'text_\']]\n\n# Calculate percentiles from 90 to 99\npercentiles = np.percentile(token_lengths, range(90, 100))\n\n# Display results\nfor i, p in enumerate(percentiles, start=90):\n    print(f"{i}th percentile: {int(p)} tokens")\n\n# Or get specific percentiles\np90 = np.percentile(token_lengths, 90)\np95 = np.percentile(token_lengths, 95)\np99 = np.percentile(token_lengths, 99)\n\nprint(f"\n90th percentile: {int(p90)}")\nprint(f"95th percentile: {int(p95)}")\nprint(f"99th percentile: {int(p99)}")'

In [10]:
# Split into training and validation sets
dataset = dataset['train'].train_test_split(test_size=0.2, seed=seed_value)
dataset_test = dataset['test'].train_test_split(test_size=0.7, seed=seed_value)

In [11]:
dataset_test

DatasetDict({
    train: Dataset({
        features: ['category', 'rating', 'label', 'text_'],
        num_rows: 2426
    })
    test: Dataset({
        features: ['category', 'rating', 'label', 'text_'],
        num_rows: 5661
    })
})

In [12]:
dataset_test['test'][1998]

{'category': 'Electronics_5',
 'rating': 1.0,
 'label': 'CG',
 'text_': 'not worth the money. not what I expected.Worst item in the world for the price.'}

In [13]:
from datasets import DatasetDict
dataset_train = DatasetDict({'train': dataset['train'], 'validation': dataset_test['train'], 'test': dataset_test['test']})

In [14]:
dataset_train

DatasetDict({
    train: Dataset({
        features: ['category', 'rating', 'label', 'text_'],
        num_rows: 32345
    })
    validation: Dataset({
        features: ['category', 'rating', 'label', 'text_'],
        num_rows: 2426
    })
    test: Dataset({
        features: ['category', 'rating', 'label', 'text_'],
        num_rows: 5661
    })
})

In [15]:
def tokenize_function(examples):
    return tokenizer(examples["text_"], padding=True, truncation=True)

def map_labels_to_ids(examples):
    examples["label"] = [label2id[label] for label in examples["label"]]
    return examples

# Apply the tokenizer to the dataset
train_dataset = dataset_train['train'].map(tokenize_function, batched=True)
val_dataset = dataset_train["validation"].map(tokenize_function, batched=True)
test_dataset = dataset_train["test"].map(tokenize_function, batched=True)

# Map string labels to integer IDs
train_dataset = train_dataset.map(map_labels_to_ids, batched=True)
val_dataset = val_dataset.map(map_labels_to_ids, batched=True)
test_dataset = test_dataset.map(map_labels_to_ids, batched=True)

Map:   0%|          | 0/32345 [00:00<?, ? examples/s]

Map:   0%|          | 0/2426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5661 [00:00<?, ? examples/s]

Map:   0%|          | 0/32345 [00:00<?, ? examples/s]

Map:   0%|          | 0/2426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5661 [00:00<?, ? examples/s]


We compute **class weights** using scikit-learn’s ```compute_class_weight```.  
This is useful when training on datasets where certain classes are underrepresented, ensuring the model does not become biased towards majority labels.

In [16]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

labels = train_dataset["label"]
class_weights = compute_class_weight("balanced", classes=np.unique(labels), y=labels)

In [17]:
# We rename the dataset column from **`label`** to **`labels`**, since this is the expected field name for Hugging Face `Trainer`.
train_dataset = train_dataset.rename_column("label", "labels")
val_dataset = val_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

train_dataset = train_dataset.rename_column("text_", "text")
val_dataset = val_dataset.rename_column("text_", "text")
test_dataset = test_dataset.rename_column("text_", "text")

In [18]:
train_dataset

Dataset({
    features: ['category', 'rating', 'labels', 'text', 'input_ids', 'attention_mask'],
    num_rows: 32345
})

In [19]:
class_weights

array([1.00350583, 0.99651858])


We define a `compute_metrics` function to evaluate the model during training.  
Here we use **accuracy** from scikit-learn, which compares predicted labels with the ground truth.  

**[NOTE]** Accuracy is a good baseline, but for imbalanced datasets you may also want to track metrics like **F1-score**, **precision**, or **recall**.  


In [20]:
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)

    # Calculate confusion matrix
    cm = confusion_matrix(labels, preds)
    print("\nConfusion Matrix:\n")
    print(cm)

    metrics = {"accuracy": accuracy_score(labels, preds)}

    # Calculate precision, recall, f1-score for each class
    # The 'average=None' ensures scores are returned for each class
    # We use labels=np.unique(labels) to ensure metrics are calculated for all present labels
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average=None, labels=np.unique(labels))

    # Log metrics for each class, using id2label for descriptive names
    for i, label_name in id2label.items():
        metrics[f"precision_{label_name}"] = precision[i]
        metrics[f"recall_{label_name}"] = recall[i]
        metrics[f"f1_{label_name}"] = f1[i]

    return metrics

In [21]:
"""from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Get predictions on the test dataset
predictions = trainer.predict(test_dataset)

# Extract predicted labels and true labels
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Generate classification report
report = classification_report(true_labels, predicted_labels, target_names=list(id2label.values()))
print("\nClassification Report:\n")
print(report)

# Generate confusion matrix
cm = confusion_matrix(true_labels, predicted_labels)
print("\nConfusion Matrix:\n")
print(cm)"""

'from sklearn.metrics import classification_report, confusion_matrix\nimport numpy as np\n\n# Get predictions on the test dataset\npredictions = trainer.predict(test_dataset)\n\n# Extract predicted labels and true labels\npredicted_labels = np.argmax(predictions.predictions, axis=1)\ntrue_labels = predictions.label_ids\n\n# Generate classification report\nreport = classification_report(true_labels, predicted_labels, target_names=list(id2label.values()))\nprint("\nClassification Report:\n")\nprint(report)\n\n# Generate confusion matrix\ncm = confusion_matrix(true_labels, predicted_labels)\nprint("\nConfusion Matrix:\n")\nprint(cm)'

<a name="Train"></a>
### Train the model
Now let's use Huggingface  `Trainer`! More docs here: [Transformers docs](https://huggingface.co/docs/transformers/main_classes/trainer). We train for one full epoch (num_train_epochs=1) to get a meaningful result.

In [22]:
from transformers import TrainingArguments,Trainer,EarlyStoppingCallback
from unsloth import is_bfloat16_supported

trainer = Trainer(
    model = model,
    processing_class = tokenizer,
    eval_dataset = val_dataset,
    train_dataset = train_dataset,
    args = TrainingArguments(
        per_device_train_batch_size = 32,
        gradient_accumulation_steps = 2,
        warmup_ratio = 0.06,
        num_train_epochs = 6,  # Increase from 2
        learning_rate = 5e-6,  # Reduce from 2e-5 for more stable training
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.03,  # Increase from 0.01 for stronger regularization
        eval_strategy = "steps",
        eval_steps = 50,
        max_grad_norm=1.0,
        label_smoothing_factor=0.01,
        lr_scheduler_type = "cosine_with_restarts",  # Change from linear for smoother decay
        lr_scheduler_kwargs = {"num_cycles": 2},
        seed = seed_value,
        output_dir = "outputs",
        report_to = "none",
        save_strategy = "steps",
        save_steps = 50,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_accuracy",
        greater_is_better = True,
        save_total_limit = 3,
    ),
    compute_metrics = compute_metrics,
    callbacks = [EarlyStoppingCallback(early_stopping_patience=8)],
)

Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [23]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 32,345 | Num Epochs = 6 | Total steps = 3,036
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 2 x 1) = 64
 "-____-"     Trainable parameters = 35,194,624 of 529,229,184 (6.65% trained)
Unsloth: Not an error, but Qwen2ForSequenceClassification does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss,Accuracy,Precision Or,Recall Or,F1 Or,Precision Cg,Recall Cg,F1 Cg
50,0.643800,0.811400,0.718879,0.692878,0.777038,0.732549,0.751391,0.661765,0.703736
100,0.454800,0.409632,0.846249,0.869759,0.811148,0.839432,0.826054,0.880719,0.852511
150,0.305500,0.329548,0.881286,0.834064,0.949251,0.887938,0.942344,0.814542,0.873795
200,0.240300,0.206522,0.927040,0.921118,0.932612,0.926829,0.933002,0.921569,0.927250
250,0.148400,0.163304,0.948063,0.924290,0.975042,0.948988,0.974093,0.921569,0.947103
300,0.214600,0.122999,0.967436,0.963666,0.970882,0.967261,0.971193,0.964052,0.967610
350,0.203700,0.115898,0.965787,0.953036,0.979201,0.965942,0.979009,0.952614,0.965631
400,0.087300,0.111194,0.969497,0.958537,0.980865,0.969572,0.980769,0.958333,0.969421
450,0.093000,0.147670,0.958780,0.991087,0.925125,0.956971,0.930982,0.991830,0.960443
500,0.068400,0.099798,0.975680,0.981466,0.969218,0.975303,0.970137,0.982026,0.976045



Confusion Matrix:

[[934 268]
 [414 810]]

Confusion Matrix:

[[ 975  227]
 [ 146 1078]]

Confusion Matrix:

[[1141   61]
 [ 227  997]]

Confusion Matrix:

[[1121   81]
 [  96 1128]]

Confusion Matrix:

[[1172   30]
 [  96 1128]]

Confusion Matrix:

[[1167   35]
 [  44 1180]]

Confusion Matrix:

[[1177   25]
 [  58 1166]]

Confusion Matrix:

[[1179   23]
 [  51 1173]]

Confusion Matrix:

[[1112   90]
 [  10 1214]]

Confusion Matrix:

[[1165   37]
 [  22 1202]]

Confusion Matrix:

[[1175   27]
 [  29 1195]]

Confusion Matrix:

[[1183   19]
 [  38 1186]]

Confusion Matrix:

[[1181   21]
 [  38 1186]]

Confusion Matrix:

[[1156   46]
 [  15 1209]]

Confusion Matrix:

[[1164   38]
 [  17 1207]]

Confusion Matrix:

[[1173   29]
 [  17 1207]]

Confusion Matrix:

[[1185   17]
 [  35 1189]]

Confusion Matrix:

[[1174   28]
 [  20 1204]]

Confusion Matrix:

[[1189   13]
 [  31 1193]]

Confusion Matrix:

[[1190   12]
 [  27 1197]]

Confusion Matrix:

[[1171   31]
 [  17 1207]]

Confusion Matrix

<a name="Inference"></a>
### Inference
Let's run the model !

In [24]:
from transformers import pipeline

idx = 1998

sentence1 = test_dataset['text'][idx]

classifier = pipeline("sentiment-analysis", model=model,tokenizer=tokenizer)

print(id2label[test_dataset['labels'][idx]])
classifier(sentence1)

Device set to use cuda:0


CG


[{'label': 'CG', 'score': 0.9866234660148621}]

In [25]:
eval_results = trainer.evaluate(test_dataset)
print(eval_results)


Confusion Matrix:

[[2861   37]
 [  52 2711]]
{'eval_loss': 0.07711797207593918, 'eval_accuracy': 0.984278396043102, 'eval_precision_OR': 0.9821489872983179, 'eval_recall_OR': 0.9872325741890959, 'eval_f1_OR': 0.9846842195835485, 'eval_precision_CG': 0.9865356622998545, 'eval_recall_CG': 0.9811798769453492, 'eval_f1_CG': 0.9838504808564689, 'eval_runtime': 157.0096, 'eval_samples_per_second': 36.055, 'eval_steps_per_second': 4.509, 'epoch': 2.7675568743818}


<a name="Save"></a>
### Saving finetuned models
To save the final model, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.


In [26]:
model.save_pretrained("model")  # Local saving
tokenizer.save_pretrained("model")
# model.push_to_hub("your_name/model", token = "...") # Online saving
# tokenizer.push_to_hub("your_name/model", token = "...") # Online saving

('model/tokenizer_config.json',
 'model/special_tokens_map.json',
 'model/chat_template.jinja',
 'model/vocab.json',
 'model/merges.txt',
 'model/added_tokens.json',
 'model/tokenizer.json')

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>


In [27]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import torch

print("\n--- Loading new model: KandirResearch/ModernBERT-large-fake-review-detection ---\n")

# Load the new model and tokenizer directly using transformers, bypassing Unsloth's FastModel
# This is done to avoid the RuntimeError: Detected that you are using FX to symbolically trace a dynamo-optimized function.
model_name_to_load = "KandirResearch/ModernBERT-large-fake-review-detection"

tokenizer_new = AutoTokenizer.from_pretrained(model_name_to_load)

# Load the model in full precision first to avoid potential conflicts with unsloth's 4bit handling.
# If memory becomes an issue, consider adding a quantization_config from transformers or manually setting fp16/bf16.
model_new = AutoModelForSequenceClassification.from_pretrained(
    model_name_to_load,
    num_labels=2, # Assuming 2 labels based on id2label and label2id from kernel state
    id2label=id2label, # Reusing id2label from kernel state
    label2id=label2id, # Reusing label2id from kernel state
)

# Move model to GPU if available
if torch.cuda.is_available():
    model_new = model_new.to("cuda")

# Define minimal TrainingArguments for evaluation
eval_args = TrainingArguments(
    output_dir="./eval_results_new_model", # New output directory for this model's results
    per_device_eval_batch_size=8, # Reduced batch size to prevent OOM
    report_to="none",
    # If memory still an issue, uncomment these for fp16/bf16 if your GPU supports it:
    # fp16 = True, # or bf16 = True
)

# Instantiate a new Trainer for evaluation with the new model
trainer_eval = Trainer(
    model=model_new,
    tokenizer=tokenizer_new,
    args=eval_args,
)

print("\n--- Running predictions on the test dataset ---\n")

# Get predictions on the test dataset (test_dataset is available from previous cells)
predictions_new = trainer_eval.predict(test_dataset)

# Extract predicted labels and true labels
predicted_labels_new = np.argmax(predictions_new.predictions, axis=1)
true_labels_new = predictions_new.label_ids

# Generate classification report
report_new = classification_report(true_labels_new, predicted_labels_new, target_names=list(id2label.values()))
print("\nClassification Report for KandirResearch/ModernBERT-large-fake-review-detection:\n")
print(report_new)

# Generate confusion matrix
cm_new = confusion_matrix(true_labels_new, predicted_labels_new)
print("\nConfusion Matrix for KandirResearch/ModernBERT-large-fake-review-detection:\n")
print(cm_new)



--- Loading new model: KandirResearch/ModernBERT-large-fake-review-detection ---



tokenizer_config.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.58M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at KandirResearch/ModernBERT-large-fake-review-detection and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_20/474781562.py:37: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer._unsloth___init__`. Use `processing_class` instead.
  trainer_eval = Trainer(



--- Running predictions on the test dataset ---



unknown:0: unknown: block: [78,0,0], thread: [192,0,0] Assertion `index out of bounds: 0 <= tmp4 < 50368` failed.
unknown:0: unknown: block: [78,0,0], thread: [193,0,0] Assertion `index out of bounds: 0 <= tmp4 < 50368` failed.
unknown:0: unknown: block: [78,0,0], thread: [194,0,0] Assertion `index out of bounds: 0 <= tmp4 < 50368` failed.
unknown:0: unknown: block: [78,0,0], thread: [195,0,0] Assertion `index out of bounds: 0 <= tmp4 < 50368` failed.
unknown:0: unknown: block: [78,0,0], thread: [196,0,0] Assertion `index out of bounds: 0 <= tmp4 < 50368` failed.
unknown:0: unknown: block: [78,0,0], thread: [197,0,0] Assertion `index out of bounds: 0 <= tmp4 < 50368` failed.
unknown:0: unknown: block: [78,0,0], thread: [198,0,0] Assertion `index out of bounds: 0 <= tmp4 < 50368` failed.
unknown:0: unknown: block: [78,0,0], thread: [199,0,0] Assertion `index out of bounds: 0 <= tmp4 < 50368` failed.
unknown:0: unknown: block: [78,0,0], thread: [200,0,0] Assertion `index out of bounds: 0

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
